In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import autoaugment
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed()

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [8]:
class SkinCancerDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.file_list = list(self.root_dir.glob('*.jpg'))
        self.transform = transform
        
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        image = Image.open(img_path)
        label = int(img_path.name.split('_')[0]) - 1  # Convert to 0-based index
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Data Augmentation and Normalization
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),  # More aggressive cropping
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    autoaugment.TrivialAugmentWide(),  # Stronger augmentations
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])
])

In [9]:
# Paths configuration
data_path = Path('../Project')

# Create datasets
train_dataset = SkinCancerDataset(data_path/'train', transform=train_transform)
val_dataset = SkinCancerDataset(data_path/'val', transform=val_transform)

# Handle class imbalance
train_labels = [int(f.name.split('_')[0])-1 for f in train_dataset.file_list]
class_counts = np.bincount(train_labels)
epsilon = 1e-5
class_weights = 1. / (class_counts + epsilon)
class_weights = class_weights / class_weights.sum()  # Normalize
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True,persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=4, pin_memory=True,persistent_workers=True)


In [10]:
class ModifiedMobileNet(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        # Load pretrained MobileNetV3
        self.base = models.mobilenet_v3_large(pretrained=True)
        for param in self.base.parameters():
            param.requires_grad = False
        
        # Modify first convolution for grayscale input
        original_first_conv = self.base.features[0][0]
        self.base.features[0][0] = nn.Conv2d(
            1, original_first_conv.out_channels,
            kernel_size=original_first_conv.kernel_size,
            stride=original_first_conv.stride,
            padding=original_first_conv.padding,
            bias=False
        )
        
        # Modify classifier
        self.base.classifier[-1] = nn.Linear(
            self.base.classifier[-1].in_features, num_classes)
        
        # Freeze early layers
        for param in self.base.features[:5].parameters():
            param.requires_grad = False
    def unfreeze_layers(self, epoch):
        # Unfreeze deeper layers progressively
        if epoch >= 10:
            for param in self.base.features[5:].parameters():
                param.requires_grad = True
        if epoch >= 20:
            for param in self.base.classifier.parameters():
                param.requires_grad = True
    def forward(self, x):
        return self.base(x)

model = ModifiedMobileNet().to(device)

In [11]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha  # Should be your class weights tensor
        self.gamma = gamma

    def forward(self, x, target):
        ce_loss = F.cross_entropy(x, target, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        loss = (1 - pt) ** self.gamma * ce_loss
        return loss.mean()

In [12]:
class_weights = class_weights.to(device)
criterion = FocalLoss(alpha=class_weights, gamma=2)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = OneCycleLR(
    optimizer, 
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=30,
    pct_start=0.3  # Longer warmup
)

# Mixed precision training
scaler = GradScaler()

In [ ]:
best_val_acc = 0.0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(30):
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    model.unfreeze_layers(epoch)
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Mixed precision forward
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Backward and optimize
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()
        
        # Metrics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Print epoch summary
    print(f'Epoch [{epoch+1}/30]')
    print(f'Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%')
    print(classification_report(all_labels, all_preds, target_names=[
        'akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'], zero_division=0))
    print('-' * 60)


In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train')
plt.plot(val_accs, label='Validation')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.show()

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
           yticklabels=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
def test_model(test_dir):
    # Load best model
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()
    
    # Create test dataset
    test_transform = val_transform  # Same as validation
    test_dataset = SkinCancerDataset(test_dir, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    # Evaluation
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Metrics
    print(classification_report(all_labels, all_preds, target_names=[
        'akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'], zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
               yticklabels=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Test Confusion Matrix')
    plt.show()
